# The massive Endpoint test for debugging while serving

In [ ]:
HOST = 'http://0.0.0.0'
PORT = '8765'
URL  = '%s:%s' % (HOST, PORT)

In [ ]:
import json
import urllib.request


class RemoteEndpoint:
	def __init__(self, base_url = URL):
		self.base_url = base_url

		self.capabilities = self.get_capabilities()
		self.functions	  = self.get_functions()


	def get_capabilities(self):
		with urllib.request.urlopen('%s/meta' % self.base_url) as response:
			meta = json.load(response)

		capabilities = meta.get('capabilities')

		if capabilities is None:
			raise RuntimeError('The Endpoint metadata does not expose capabilities.')

		return capabilities


	def get_functions(self):
		functions = {}

		for cap in self.capabilities:
			assert cap['type'] == 'function'

			key = cap['function']['name']
			dsc = cap['function']['description']

			assert cap['function']['parameters']['type'] == 'object'

			req = cap['function']['parameters'].get('required', [])

			args = {}

			for nam, val in cap['function']['parameters']['properties'].items():
				args[nam] = {'typ': val['type'], 'dsc': val.get('description', ''), 'req': nam in req}

			functions[key] = {'dsc': dsc, 'args': args}

		return functions


	def run(self, fun_name, args, easy = True):
		fun = self.functions[fun_name]

		if easy:	# Help the caller by trying to match the provided arguments to the function's expected arguments.

			if type(args) is str and len(fun['args']) == 1:
				key, val = next(iter(fun['args'].items()))
				if val['typ'] == 'string':
					args = {key: args}

		data = json.dumps({'name': fun_name, 'arguments': args}).encode('utf-8')

		req = urllib.request.Request('%s/run' % self.base_url, data = data, headers = {'content-type': 'application/json'})

		with urllib.request.urlopen(req) as response:
			result = json.load(response)

		if easy:	# Help the caller when the result is a simple message after completion.

			if type(result) is dict and len(result) == 2 and 'finish_reason' in result and 'message' in result:
				if result['finish_reason'] == 'stop':
					return result['message']

		return result


	def dry_run(self, fun_name, args):
		fun = self.functions[fun_name]

		data = json.dumps({'name': fun_name, 'arguments': args}).encode('utf-8')

		req = urllib.request.Request('%s/dry_run' % self.base_url, data = data, headers = {'content-type': 'application/json'})

		with urllib.request.urlopen(req) as response:
			result = json.load(response)

		return result


In [ ]:
ep = RemoteEndpoint(URL)

In [ ]:
ep.functions

In [ ]:
# ep.capabilities

In [ ]:
ep.run('children_by_idx_markdown_example', '')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example')

In [ ]:
ep.run('children_by_idx_markdown_example', 'markdown_example|more/README.md')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example|more/README.md')

In [ ]:
ep.run('children_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1')

In [ ]:
ep.run('children_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1')

In [ ]:
ep.run('children_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1')

In [ ]:
ep.run('children_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1|header_4_1')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1|header_4_1')

In [ ]:
ep.run('children_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1|header_4_1|paragraph_3')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1|header_4_1|paragraph_3')

In [ ]:
ep.run('children_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1|header_4_1|paragraph_3|text_7')

In [ ]:
ep.run('object_by_idx_markdown_example', 'markdown_example|more/README.md|header_1_1|header_2_1|header_3_1|header_4_1|paragraph_3|text_7')

In [ ]:
ep.run('children_by_idx_wikipedia_2025', '')

In [ ]:
ep.run('object_by_idx_wikipedia_2025', 'wikipedia_2025')

In [ ]:
ep.run('children_by_idx_wikipedia_2025', 'wikipedia_2025|1:42')

In [ ]:
ep.run('object_by_idx_wikipedia_2025', 'wikipedia_2025|1:42')

In [ ]:
ep.run('children_by_idx_wikipedia_2025', 'wikipedia_2025|1:42|Politics of Switzerland.md')

In [ ]:
ep.run('object_by_idx_wikipedia_2025', 'wikipedia_2025|1:42|Politics of Switzerland.md')

In [ ]:
ep.run('children_by_idx_wikipedia_2025', 'wikipedia_2025|1:42|Politics of Switzerland.md|header_1_1')

In [ ]:
ep.run('object_by_idx_wikipedia_2025', 'wikipedia_2025|1:42|Politics of Switzerland.md|header_1_1')

In [ ]:
ep.run('children_by_idx_russell_works', '')

In [ ]:
ep.run('object_by_idx_russell_works', 'russell_works')

In [ ]:
ep.run('children_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf')

In [ ]:
ep.run('object_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf')

In [ ]:
ep.run('children_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf|header_1_1')

In [ ]:
ep.run('object_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf|header_1_1')

In [ ]:
ep.run('children_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf|header_1_1|paragraph_5')

In [ ]:
ep.run('object_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf|header_1_1|paragraph_5')

In [ ]:
ep.run('children_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf|header_1_1|paragraph_5|text_6')

In [ ]:
ep.run('object_by_idx_russell_works', 'russell_works|Russell, The Problems of Philosophy.pdf|header_1_1|paragraph_5|text_6')

In [ ]:
ep.dry_run('children_by_idx_russell_works', {})

In [ ]:
ep.dry_run('children_by_idx_russell_works', {'index': 'russell_works'})

In [ ]:
ep.run('children_by_idx_entities', '')

In [ ]:
ep.run('children_by_idx_entities', 'entities|person')

In [ ]:
ep.run('children_by_idx_entities', 'entities|person|ceo')

In [ ]:
ep.run('children_by_idx_entities', 'entities|company')

In [ ]:
ep.run('children_by_idx_entities', 'entities|company|germany')

In [ ]:
ep.run('children_by_idx_entities', 'entities|company|germany|chemical')

In [ ]:
ep.run('children_by_idx_entities', 'entities|country')

In [ ]:
ep.run('node_by_idx_entities', 'entities|person')

In [ ]:
ep.run('node_by_idx_entities', 'entities|company')

In [ ]:
ep.run('node_by_idx_entities', 'entities|country')

In [ ]:
ep.run('node_by_idx_entities', 'entities|person|ceo')

In [ ]:
ep.run('node_by_idx_entities', 'entities|company|germany')

In [ ]:
ep.run('node_by_idx_entities', 'entities|company|germany|chemical')

In [ ]:
ep.run('children_by_idx_relationships', '')

In [ ]:
ep.run('children_by_idx_relationships', 'relationships|is_ceo')

In [ ]:
ep.run('children_by_idx_relationships', 'relationships|is_ceo|german_company')

In [ ]:
ep.run('children_by_idx_relationships', 'relationships|company_in_country')

In [ ]:
ep.run('node_by_idx_relationships', 'relationships|is_ceo')

In [ ]:
ep.run('node_by_idx_relationships', 'relationships|is_ceo|german_company')

In [ ]:
ep.run('node_by_idx_relationships', 'relationships|company_in_country')

In [ ]:
ep.run('children_by_idx_known_ids', '')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|person')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|person|Fritz')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|person|Otto')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|company')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|company|bar')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|company|germany')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|company|germany|chemical')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|company|germany|chemical|MeinBier')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|country')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|country|Germany')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|_edge_')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|_edge_|id_fritz_ceo_barx')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|_edge_|id_otto_ceo_g_mb')

In [ ]:
ep.run('children_by_idx_known_ids', 'known_ids|_edge_|id_mb_in_g')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|person')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|person|Fritz')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|person|Otto')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|company')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|company|bar')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|company|germany|chemical|MeinBier')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|country')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|country|Germany')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|_edge_')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|_edge_|id_fritz_ceo_bar')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|_edge_|id_otto_ceo_g_mb')

In [ ]:
ep.run('node_by_idx_known_ids', 'known_ids|_edge_|id_mb_in_g')